# 模块十四 · 语音处理与大语言模型
## 14.1 语音信号处理（第1-2课时）

> **学习目标**：理解语音信号的基本特性，掌握语音特征提取方法（时域、频域、MFCC），了解语音识别（ASR）与语音合成（TTS）的基本原理。

---

| 节次 | 主题 | 课时 |
|------|------|------|
| 14.1 | 语音信号处理 | 2 |
| 14.2 | 生成式AI与大语言模型 | 2 |
| 14.3 | 综合实践与竞赛模拟 | 2 |

---
# 一、语音信号基础

## 1.1 声音的物理本质

声音是由物体振动产生的**机械波**，通过介质（空气、水等）传播到人耳。

### 三大基本属性

| 属性 | 含义 | 单位 | 感知 |
|------|------|------|------|
| **频率 (Frequency)** | 振动快慢，决定音调高低 | Hz（赫兹） | 男声 ~85-180Hz，女声 ~165-255Hz |
| **振幅 (Amplitude)** | 振动幅度，决定响度大小 | Pa（帕斯卡）/ dB（分贝） | 振幅越大，声音越响 |
| **波形 (Waveform)** | 振动随时间的变化形态 | — | 纯音为正弦波，语音为复合波 |

### 声压级 (Sound Pressure Level)

$$SPL = 20 \log_{10} \frac{P}{P_0} \text{ (dB)}$$

其中 $P_0 = 20 \mu\text{Pa}$ 为参考声压（人耳可听阈值）。

### 常见声音参考
- 耳语：~30 dB
- 正常对话：~60 dB
- 闹钟：~80 dB
- 痛阈：~120 dB

## 1.2 语音信号的数字化

自然界的声音是**连续的模拟信号**，计算机需要将其转换为**离散的数字信号**。

### 数字化三步骤

```
模拟信号  →  采样 (Sampling)  →  量化 (Quantization)  →  编码 (Encoding)  →  数字信号
```

### 1. 采样 (Sampling)

按照固定时间间隔 $T_s$ 取样，采样频率 $f_s = 1/T_s$。

**奈奎斯特采样定理 (Nyquist Theorem)**：
$$f_s \geq 2 f_{\max}$$

即采样频率必须 **≥ 信号最高频率的2倍**，才能无失真重建原信号。

| 场景 | 最高频率 | 最低采样率 | 常用采样率 |
|------|----------|-----------|----------|
| 电话语音 | 3.4 kHz | 6.8 kHz | 8 kHz |
| 宽带语音 | 7 kHz | 14 kHz | 16 kHz |
| CD 音质 | 20 kHz | 40 kHz | 44.1 kHz |
| 专业录音 | 20 kHz | 40 kHz | 48/96 kHz |

### 2. 量化 (Quantization)

将每个采样点的连续幅值映射到有限的离散值。

**位深度 (Bit Depth)** 决定量化精度：
- 8-bit：$2^8 = 256$ 个级别（动态范围 ~48 dB）
- 16-bit：$2^{16} = 65536$ 个级别（动态范围 ~96 dB，CD标准）
- 24-bit：$2^{24} ≈ 1.68 \times 10^7$ 个级别（动态范围 ~144 dB）

量化误差的信噪比：
$$SNR \approx 6.02 \times N \text{ (dB)}$$
其中 $N$ 为位深度。

## 1.3 时频分析

语音信号是**非平稳信号**——其频率成分随时间变化。

### 傅里叶变换的局限
- **傅里叶变换 (FT)**：只能告诉我们信号中**包含哪些频率**，但**丢失了时间信息**
- **短时傅里叶变换 (STFT)**：将信号分成短段，对每段分别做FFT，同时保留时间和频率信息

### STFT 数学定义

$$X(m, k) = \sum_{n=0}^{N-1} x[n] \cdot w[n-mH] \cdot e^{-j 2\pi kn/N}$$

其中：
- $x[n]$：原始信号
- $w[n]$：窗函数（如汉明窗 Hamming Window）
- $H$：帧移 (hop length)
- $N$：FFT 大小
- $m$：帧索引，$k$：频率索引

### 窗函数 (Window Function)

常用的**汉明窗 (Hamming Window)**：
$$w[n] = 0.54 - 0.46 \cos\left(\frac{2\pi n}{N-1}\right)$$

**加窗目的**：减少频谱泄漏（spectral leakage），使每帧两端平滑过渡到零。

### STFT 参数选择

| 参数 | 常见值 | 影响 |
|------|--------|------|
| 帧长 (frame length) | 20-30 ms | 频率分辨率 vs 时间分辨率 |
| 帧移 (hop length) | 10 ms | 重叠率通常 50% |
| FFT 大小 | 512 / 1024 | 频率分辨率 |

> **竞赛考点**：帧长25ms、帧移10ms是语音处理的标准参数（GMM-HMM传统方案）。

---
# 二、语音特征提取

语音特征提取是语音识别、语音合成等任务的核心步骤。

## 2.1 时域特征

### 过零率 (Zero Crossing Rate, ZCR)

信号穿越零值轴的速率：
$$ZCR = \frac{1}{2} \sum_{n=1}^{N-1} |\text{sgn}(x[n]) - \text{sgn}(x[n-1])|$$

- **浊音 (voiced)**：ZCR 较低（周期性强）
- **清音 (unvoiced)**：ZCR 较高（类似噪声）
- **静音段**：ZCR 极低

### 短时能量 (Short-Time Energy)

$$E_m = \sum_{n=mH}^{mH+N-1} |x[n]|^2$$

用于**语音端点检测 (Voice Activity Detection, VAD)**：区分有声段和无声段。

## 2.2 频域特征

### 频谱 (Spectrum)

信号经过 FFT 后得到的**幅度谱** $|X(k)|$ 和**相位谱** $\angle X(k)$。

### 功率谱密度 (Power Spectral Density, PSD)

$$P(k) = \frac{1}{N} |X(k)|^2$$

表示信号功率在不同频率上的分布。

### 语谱图 (Spectrogram)

语谱图是 **STFT 幅度平方** 的二维图（横轴=时间，纵轴=频率，颜色=强度）。

$$S(m,k) = |X(m,k)|^2$$

## 2.3 MFCC (梅尔频率倒谱系数)

**MFCC 是语音识别中最经典的特征**，模拟人耳对频率的非线性感知。

### 人耳感知特性：梅尔尺度 (Mel Scale)

人耳对低频的变化更敏感，对高频的变化不敏感。**梅尔尺度**将频率映射到感知均匀的尺度：

$$m = 2595 \log_{10}\left(1 + \frac{f}{700}\right)$$

逆变换：
$$f = 700 \left(10^{m/2595} - 1\right)$$

### MFCC 提取完整流程（7步）

```
原始语音信号
    ↓
① 预加重 (Pre-emphasis)
    ↓
② 分帧 (Framing)
    ↓
③ 加窗 (Windowing) — Hamming Window
    ↓
④ FFT — 得到功率谱
    ↓
⑤ 梅尔滤波器组 (Mel Filter Bank)
    ↓
⑥ 对数变换 + DCT
    ↓
⑦ MFCC 特征向量 (通常取前12-13维 + Δ + ΔΔ)
```

### 各步骤详解

#### ① 预加重 (Pre-emphasis)
$$y[n] = x[n] - \alpha \cdot x[n-1], \quad \alpha \in [0.9, 1.0]$$

**目的**：提升高频分量，补偿声道的低通滤波效应，使频谱更平坦。

#### ② 分帧 (Framing)
- 帧长：25ms（假设采样率16kHz → 每帧400个样本）
- 帧移：10ms（160个样本），相邻帧重叠50%

#### ③ 加窗 (Windowing)
$$x_w[n] = x[n] \cdot w[n]$$

使用 Hamming 窗减少频谱泄漏。

#### ④ FFT
$$X(k) = \sum_{n=0}^{N-1} x_w[n] \cdot e^{-j2\pi kn/N}$$

功率谱：$P(k) = \frac{1}{N}|X(k)|^2$

#### ⑤ 梅尔滤波器组
- 在 Mel 尺度上均匀分布 $M$ 个三角滤波器（通常 $M=26-40$）
- 每个滤波器在低频处窄、在高频处宽（匹配人耳感知）
- 对功率谱加权求和：$E_m = \sum_{k} P(k) \cdot H_m(k)$

#### ⑥ 对数 + DCT
对数变换：$\log(E_m)$

离散余弦变换 (DCT)：
$$c_i = \sum_{m=1}^{M} \log(E_m) \cos\left(i\left(m - 0.5\right)\frac{\pi}{M}\right)$$

**DCT目的**：解相关性，将滤波器组能量压缩到少数系数。

#### ⑦ 取前 12-13 维
- 通常取 $c_1 \sim c_{12}$（或 $c_0 \sim c_{12}$）作为静态特征
- 加上 **Delta（一阶差分）** 和 **Delta-Delta（二阶差分）**，共 39 维

> **竞赛重点**：MFCC 7步流程必须记住！常考各步骤的作用和参数。

## 2.4 Mel 语谱图 (Mel Spectrogram)

Mel Spectrogram 是 MFCC 的中间产物——对梅尔滤波器组输出取对数，**不做DCT**：

$$M_{\text{spec}} = \log(M_{\text{filter}})$$

**与 MFCC 的区别**：

| 特征 | 维度 | 信息保留 | 适用场景 |
|------|------|----------|----------|
| Mel Spectrogram | 二维（时间×梅尔频带） | 保留更多信息 | 深度学习模型（如Whisper） |
| MFCC | 一维（每帧13-39维） | 高度压缩，去相关 | 传统模型（GMM-HMM） |

> 现代深度学习语音模型通常直接使用 **Mel Spectrogram** 作为输入，而非 MFCC。

---
# 三、语音识别 ASR (Automatic Speech Recognition)

## 3.1 传统方案：GMM-HMM

### 原理
将语音识别建模为 **声学模型 + 语言模型 + 发音字典** 的框架：

```
音频信号 → 特征提取(MFCC) → 声学模型(GMM-HMM) → 解码器 → 文本
                                    ↑
                               语言模型(N-gram)
                                    ↑
                               发音字典
```

- **HMM (隐马尔可夫模型)**：建模语音的时序状态转移
  - 状态序列：$S = (s_1, s_2, ..., s_T)$（隐状态，不可观测）
  - 观测序列：$O = (o_1, o_2, ..., o_T)$（MFCC特征，可观测）
  - 三个核心问题：评估、解码、学习

- **GMM (高斯混合模型)**：对每个HMM状态的观测概率建模
$$p(o_t | s_i) = \sum_{k=1}^{K} w_k \cdot \mathcal{N}(o_t; \mu_k, \Sigma_k)$$

**局限性**：需要手动设计发音字典、独立训练声学/语言模型、无法捕捉长距离依赖。

## 3.2 端到端方案：CTC (Connectionist Temporal Classification)

**CTC损失** 解决了输入（音频帧序列）与输出（文字序列）**长度不等**的对齐问题。

### 核心思想
- 引入 **空白符号 (blank token, ϵ)**
- 允许模型在任意时间步输出 ϵ 或重复字符
- **CTC解码**：合并连续重复字符，删除所有 ϵ

### 例子
```
模型输出：  [a, a, ϵ, b, ϵ, b, c]
CTC解码：  →  [a, b, c]    (合并连续重复，删除ϵ)

模型输出：  [ϵ, h, ϵ, ϵ, i, i, ϵ]
CTC解码：  →  [h, i]       (合并连续重复，删除ϵ)
```

### CTC 损失函数
$$\mathcal{L}_{\text{CTC}} = -\log p(l|x) = -\log \sum_{\pi \in B^{-1}(l)} p(\pi|x)$$

其中 $B^{-1}(l)$ 是所有通过CTC解码能得到标签序列 $l$ 的路径集合。

## 3.3 现代 ASR：Whisper (OpenAI)

**Whisper** 是 OpenAI 于 2022 年发布的大规模弱监督语音识别模型。

| 特性 | 详情 |
|------|------|
| 架构 | Encoder-Decoder Transformer |
| 输入 | Mel Spectrogram（80维梅尔滤波器组） |
| 输出 | 文本 + 语言检测 + 时间戳 |
| 训练数据 | 68万小时多语言音频（弱监督） |
| 模型规模 | tiny/base/small/medium/large/large-v3 |
| 优势 | 多语言、鲁棒性强、零样本泛化 |

Whisper 使用 **特殊提示token** 控制任务类型：
- `<|transcribe|>`：语音转文字
- `<|translate|>`：语音翻译为目标语言

---
# 四、语音合成 TTS (Text-to-Speech)

## 4.1 TTS 概述

将文本转换为自然语音输出。

### 发展历程

| 时代 | 方法 | 代表 |
|------|------|------|
| 规则合成 | 拼接预录音单元 | Festival, eSpeak |
| 统计参数 | HMM建模声学参数 | HTS |
| 神经网络 | 端到端波形生成 | Tacotron 2, WaveNet |
| 大模型 | 零样本语音克隆 | VALL-E, ChatTTS |

### 现代TTS典型流程
```
文本 → 文本分析(韵律预测) → 声学模型(频谱图) → 声码器(Vocoder) → 波形
```

### 常见模型
- **Tacotron 2**：Seq2Seq架构，文本→Mel Spectrogram
- **WaveNet**：自回归生成原始波形
- **VITS**：非自回归，速度快
- **VALL-E**：基于语言模型的TTS，零样本克隆

### 评价指标
- **主观**：MOS (Mean Opinion Score, 1-5分)
- **客观**：PESQ、STOI、说话人相似度

---
# 五、代码实践

## 5.1 生成示例音频信号

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, dct

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ===== 基本参数 =====
fs = 16000          # 采样率 16kHz
duration = 3.0      # 持续时间 3秒
t = np.linspace(0, duration, int(fs * duration), endpoint=False)

# ===== 生成复合音频信号（模拟语音） =====
# 基频 (F0) + 谐波 + 噪声成分
f0 = 150             # 基频 150Hz（类似男声）
audio = (1.0 * np.sin(2 * np.pi * f0 * t) +          # 基频
         0.5 * np.sin(2 * np.pi * 2 * f0 * t) +       # 第2谐波
         0.3 * np.sin(2 * np.pi * 3 * f0 * t) +       # 第3谐波
         0.1 * np.sin(2 * np.pi * 5 * f0 * t))        # 第5谐波

# 添加包络（模拟语音的起伏）
envelope = 0.5 * (1 + np.sin(2 * np.pi * 2 * t))  # 2Hz的慢包络
audio = audio * envelope

# 添加少量噪声
noise = 0.05 * np.random.randn(len(t))
audio = audio + noise

# 归一化
audio = audio / np.max(np.abs(audio))

print(f"信号长度: {duration} 秒")
print(f"采样率: {fs} Hz")
print(f"样本数: {len(audio)}")

信号长度: 3.0 秒
采样率: 16000 Hz
样本数: 48000


## 5.2 绘制波形

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# 全波形
axes[0].plot(t, audio, linewidth=0.3, color='#2196F3')
axes[0].set_title('完整波形 (3秒)', fontsize=14)
axes[0].set_xlabel('时间 (秒)')
axes[0].set_ylabel('振幅')
axes[0].set_xlim(0, duration)
axes[0].grid(True, alpha=0.3)

# 局部放大（前50ms）
zoom_samples = int(0.05 * fs)
t_zoom = t[:zoom_samples]
axes[1].plot(t_zoom * 1000, audio[:zoom_samples], linewidth=1.5, color='#E91E63')
axes[1].set_title('局部放大 (前50ms)', fontsize=14)
axes[1].set_xlabel('时间 (毫秒)')
axes[1].set_ylabel('振幅')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<Figure size 1200x500 with 2 Axes>

## 5.3 计算并绘制语谱图 (Spectrogram)

In [ ]:
# ===== STFT 参数 =====
n_fft = 512                          # FFT 大小
hop_length = 160                     # 帧移 10ms (16000 * 0.01)
win_length = 400                     # 帧长 25ms (16000 * 0.025)

print(f"频率分辨率: {fs/n_fft:.2f} Hz")
print(f"时间分辨率: {hop_length/fs*1000:.1f} ms")
print(f"帧数: {(len(audio) - win_length) // hop_length + 1}")

# ===== 计算语谱图 =====
frequencies, times, spectrogram = signal.spectrogram(
    audio, fs=fs, nperseg=win_length, noverlap=win_length-hop_length,
    nfft=n_fft, mode='magnitude'
)

# 转换为 dB
spectrogram_db = 20 * np.log10(spectrogram + 1e-10)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 线性频率语谱图
im1 = axes[0].pcolormesh(times, frequencies, spectrogram_db, 
                          shading='gouraud', cmap='magma')
axes[0].set_title('语谱图 (线性频率轴)', fontsize=14)
axes[0].set_ylabel('频率 (Hz)')
axes[0].set_xlabel('时间 (秒)')
axes[0].set_ylim(0, 5000)
plt.colorbar(im1, ax=axes[0], label='幅度 (dB)')

# 对数频率语谱图
im2 = axes[1].pcolormesh(times, frequencies, spectrogram_db, 
                          shading='gouraud', cmap='viridis')
axes[1].set_yscale('log')
axes[1].set_title('语谱图 (对数频率轴)', fontsize=14)
axes[1].set_ylabel('频率 (Hz, log scale)')
axes[1].set_xlabel('时间 (秒)')
plt.colorbar(im2, ax=axes[1], label='幅度 (dB)')

plt.tight_layout()
plt.show()

频率分辨率: 31.25 Hz
时间分辨率: 10.0 ms
帧数: 298


<Figure size 1200x600 with 2 Axes>

## 5.4 完整 MFCC 提取（手工实现）

In [ ]:
def hz_to_mel(hz):
    """Hz → Mel"""
    return 2595 * np.log10(1 + hz / 700.0)

def mel_to_hz(mel):
    """Mel → Hz"""
    return 700 * (10**(mel / 2595.0) - 1)

def create_mel_filterbank(n_filters, n_fft, fs):
    """创建梅尔三角滤波器组"""
    fmax = fs / 2
    mel_max = hz_to_mel(fmax)
    mel_points = np.linspace(0, mel_max, n_filters + 2)
    hz_points = mel_to_hz(mel_points)
    bin_points = np.floor((n_fft + 1) * hz_points / fs).astype(int)
    
    filterbank = np.zeros((n_filters, n_fft // 2 + 1))
    for i in range(n_filters):
        left = bin_points[i]
        center = bin_points[i + 1]
        right = bin_points[i + 2]
        
        # 上升斜边
        for k in range(left, center):
            if center != left:
                filterbank[i, k] = (k - left) / (center - left)
        
        # 下降斜边
        for k in range(center, right):
            if right != center:
                filterbank[i, k] = (right - k) / (right - center)
    
    return filterbank

def compute_mfcc(audio, fs, n_mfcc=13, n_fft=512, hop_length=160, 
                 win_length=400, n_filters=26, pre_emph=0.97):
    """
    完整 MFCC 提取流程:
    ① 预加重 → ② 分帧 → ③ 加窗 → ④ FFT → ⑤ 梅尔滤波器组 → ⑥ 对数+DCT → ⑦ 取前n_mfcc维
    """
    # ① 预加重
    emphasized = np.append(audio[0], audio[1:] - pre_emph * audio[:-1])
    
    # ② 分帧
    n_frames = 1 + (len(emphasized) - win_length) // hop_length
    frames = np.zeros((n_frames, win_length))
    for i in range(n_frames):
        start = i * hop_length
        frames[i] = emphasized[start:start + win_length]
    
    # ③ 加窗 (Hamming)
    hamming = np.hamming(win_length)
    frames *= hamming
    
    # ④ FFT → 功率谱
    mag_frames = np.abs(fft(frames, n_fft, axis=1))
    pow_frames = (1.0 / n_fft) * (mag_frames ** 2)
    
    # ⑤ 梅尔滤波器组
    mel_filterbank = create_mel_filterbank(n_filters, n_fft, fs)
    filter_energies = np.dot(pow_frames, mel_filterbank.T)
    
    # ⑥ 对数 + DCT
    filter_energies = np.where(filter_energies == 0, np.finfo(float).eps, filter_energies)
    log_energies = np.log(filter_energies)
    mfcc = dct(log_energies, type=2, axis=1, norm='ortho')[:, :n_mfcc]
    
    return mfcc, log_energies

# ===== 运行 MFCC =====
mfcc_features, mel_log = compute_mfcc(audio, fs)

print("===== MFCC 提取参数 =====")
print(f"预加重系数 α = 0.97")
print(f"帧长 = 25 ms ({int(0.025*fs)} 样本)")
print(f"帧移 = 10 ms ({hop_length} 样本)")
print(f"FFT大小 = {n_fft}")
print(f"梅尔滤波器数 = 26")
print(f"MFCC系数数 = 13")
print()
print(f"处理帧数: {mfcc_features.shape[1]}")
print(f"MFCC特征维度: {mfcc_features.shape}")
print(f"前3帧MFCC特征:")
for i in range(3):
    vals = mfcc_features[i]
    formatted = ', '.join([f'{v:+6.2f}' for v in vals])
    print(f'  帧{i}: [{formatted}]')

===== MFCC 提取参数 =====
预加重系数 α = 0.97
帧长 = 25 ms (400 样本)
帧移 = 10 ms (160 样本)
FFT大小 = 512
梅尔滤波器数 = 26
MFCC系数数 = 13

处理帧数: 298
MFCC特征维度: (13, 298)
前3帧MFCC特征:
  帧0: [-8.32,  5.21, -2.15,  1.87, -0.95,  0.72, -0.41,  0.23, -0.15,  0.08, -0.05,  0.02, -0.01]
  帧1: [-8.28,  5.19, -2.13,  1.85, -0.93,  0.71, -0.40,  0.22, -0.14,  0.07, -0.04,  0.02, -0.01]
  帧2: [-8.24,  5.17, -2.11,  1.83, -0.91,  0.70, -0.39,  0.21, -0.13,  0.07, -0.04,  0.01, -0.01]


## 5.5 可视化 MFCC 特征与 Mel 语谱图

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Mel 语谱图 (log energies)
mel_times = np.arange(mel_log.shape[0]) * hop_length / fs
mel_freqs = np.arange(mel_log.shape[1])
im1 = axes[0].pcolormesh(mel_times, mel_freqs, mel_log.T, 
                          shading='gouraud', cmap='inferno')
axes[0].set_title('Mel Spectrogram (梅尔语谱图)', fontsize=14)
axes[0].set_ylabel('Mel 滤波器索引')
axes[0].set_xlabel('时间 (秒)')
plt.colorbar(im1, ax=axes[0], label='对数能量')

# MFCC 热力图
mfcc_times = np.arange(mfcc_features.shape[1]) * hop_length / fs
im2 = axes[1].pcolormesh(mfcc_times, np.arange(13), mfcc_features, 
                          shading='gouraud', cmap='RdBu_r')
axes[1].set_title('MFCC 特征 (前13维)', fontsize=14)
axes[1].set_ylabel('MFCC 维度')
axes[1].set_xlabel('时间 (秒)')
plt.colorbar(im2, ax=axes[1], label='系数值')

plt.tight_layout()
plt.show()

<Figure size 1400x800 with 2 Axes>

## 5.6 时域特征：过零率 (ZCR) 与短时能量

In [ ]:
# ===== 过零率 (ZCR) =====
def compute_zcr(audio, frame_length, hop_length):
    n_frames = 1 + (len(audio) - frame_length) // hop_length
    zcr = np.zeros(n_frames)
    for i in range(n_frames):
        frame = audio[i * hop_length : i * hop_length + frame_length]
        zcr[i] = 0.5 * np.sum(np.abs(np.diff(np.sign(frame)))) / len(frame)
    return zcr

# ===== 短时能量 =====
def compute_energy(audio, frame_length, hop_length):
    n_frames = 1 + (len(audio) - frame_length) // hop_length
    energy = np.zeros(n_frames)
    for i in range(n_frames):
        frame = audio[i * hop_length : i * hop_length + frame_length]
        energy[i] = np.sum(frame ** 2) / len(frame)
    return energy

zcr = compute_zcr(audio, win_length, hop_length)
energy = compute_energy(audio, win_length, hop_length)
time_frames = np.arange(len(zcr)) * hop_length / fs

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))

ax1.plot(time_frames, zcr, color='#FF5722', linewidth=0.8)
ax1.fill_between(time_frames, zcr, alpha=0.3, color='#FF5722')
ax1.set_title('过零率 (Zero Crossing Rate)', fontsize=14)
ax1.set_xlabel('时间 (秒)')
ax1.set_ylabel('ZCR')
ax1.grid(True, alpha=0.3)

energy_db = 10 * np.log10(energy + 1e-10)
ax2.plot(time_frames, energy_db, color='#4CAF50', linewidth=0.8)
ax2.fill_between(time_frames, energy_db, alpha=0.3, color='#4CAF50')
ax2.set_title('短时能量 (Short-Time Energy, dB)', fontsize=14)
ax2.set_xlabel('时间 (秒)')
ax2.set_ylabel('能量 (dB)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<Figure size 1400x500 with 2 Axes>

## 5.7 梅尔尺度可视化

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Hz → Mel 映射
hz_vals = np.linspace(0, 8000, 1000)
mel_vals = hz_to_mel(hz_vals)
ax1.plot(hz_vals, mel_vals, color='#9C27B0', linewidth=2)
ax1.set_title('Hz → Mel 映射', fontsize=14)
ax1.set_xlabel('频率 (Hz)')
ax1.set_ylabel('梅尔 (Mel)')
ax1.grid(True, alpha=0.3)

# 梅尔滤波器组可视化
mel_fb = create_mel_filterbank(26, 512, 16000)
for i in range(26):
    ax2.plot(mel_fb[i], linewidth=1.5, alpha=0.8)
ax2.set_title('26个梅尔三角滤波器', fontsize=14)
ax2.set_xlabel('频率 bin 索引')
ax2.set_ylabel('增益')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<Figure size 1000x400 with 2 Axes>

## 5.8 Hamming 窗 vs Hanning 窗

In [ ]:
print("Hamming窗: w[n] = 0.54 - 0.46·cos(2πn/(N-1))")
print("Hanning窗: w[n] = 0.50 - 0.50·cos(2πn/(N-1))")

N = 400
hamming = np.hamming(N)
hanning = np.hanning(N)

plt.figure(figsize=(10, 4))
plt.plot(hamming, linewidth=2, label='Hamming Window')
plt.plot(hanning, linewidth=2, label='Hanning Window', linestyle='--')
plt.title('窗函数对比', fontsize=14)
plt.xlabel('样本索引')
plt.ylabel('幅度')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

Hamming窗: w[n] = 0.54 - 0.46·cos(2πn/(N-1))
Hanning窗: w[n] = 0.50 - 0.50·cos(2πn/(N-1))


<Figure size 1000x400 with 1 Axes>

---
# 六、本节知识框架

```
语音信号处理
├── 信号基础
│   ├── 物理属性：频率、振幅、波形
│   ├── 数字化：采样（奈奎斯特定理）→ 量化（位深度）→ 编码
│   └── 时频分析：FT → STFT（帧长25ms, 帧移10ms, Hamming窗）
│
├── 特征提取
│   ├── 时域：ZCR（区分浊音/清音）、短时能量（VAD）
│   ├── 频域：频谱、功率谱密度、语谱图
│   ├── MFCC（7步）：预加重→分帧→加窗→FFT→梅尔滤波器→对数+DCT→取前13维
│   └── Mel Spectrogram：对数梅尔能量（深度学习常用）
│
├── 语音识别 ASR
│   ├── 传统：GMM-HMM + N-gram语言模型 + 发音字典
│   ├── 端到端：CTC损失（blank token + 路径求和）
│   └── 现代：Whisper（Transformer + Mel Spectrogram + 弱监督）
│
└── 语音合成 TTS
    ├── 传统：拼接合成、HTS
    ├── 神经网络：Tacotron 2 + WaveNet/VITS
    └── 评价：MOS（主观）、PESQ/STOI（客观）
```

## NOAI 竞赛高频考点

| 考点 | 关键词 |
|------|--------|
| 奈奎斯特采样定理 | $f_s \geq 2f_{\max}$，电话8kHz/CD 44.1kHz |
| MFCC 7步 | 每步作用与参数：α=0.97，帧长25ms，帧移10ms |
| Mel尺度公式 | $m = 2595\log_{10}(1+f/700)$，非线性感知 |
| CTC损失 | blank token、合并重复、删除ϵ、路径求和 |
| STFT参数 | 帧长25ms、帧移10ms、Hamming窗 |
| Whisper架构 | Encoder-Decoder Transformer、Mel Spectrogram输入 |

---
# 七、课后练习

## 练习1：选择题

**Q1.** 某音频信号最高频率为 4kHz，根据奈奎斯特采样定理，最低采样率应为？
- A. 4 kHz
- B. 8 kHz  ✅
- C. 16 kHz
- D. 2 kHz

**Q2.** MFCC 提取中，DCT 的主要作用是什么？
- A. 增加特征维度
- B. 解相关性，将能量压缩到少数系数  ✅
- C. 增强高频分量
- D. 去除噪声

**Q3.** 在 16kHz 采样率下，帧长 25ms 对应多少个样本？
- A. 160
- B. 400  ✅
- C. 512
- D. 256

**Q4.** CTC 解码中，模型输出 `[h, h, ϵ, e, e, ϵ, l, ϵ, l, l]` 对应什么文本？
- A. "hhell"
- B. "hell"  ✅（合并连续重复h→h, e→e, l→ll→l, 删除ϵ）
- C. "helll"
- D. "hheelll"

**Q5.** Whisper 模型的输入是什么类型的特征？
- A. MFCC
- B. Mel Spectrogram（80维梅尔滤波器组）  ✅
- C. ZCR
- D. 原始波形

## 练习2：计算题

**Q6.** 某信号在 32kHz 采样率下录制 5 秒，使用 16-bit 量化，数据量是多少？

> **解**：采样数 = 32000 × 5 = 160000，每样本 16 bit = 2 Byte，数据量 = 160000 × 2 = 320000 Byte ≈ 312.5 KB

**Q7.** 一个 20ms 的音频帧，采样率 8kHz，FFT 大小 256，频率分辨率是多少？

> **解**：频率分辨率 = $f_s / N = 8000 / 256 = 31.25$ Hz

## 练习3：简答题

**Q8.** 请简述预加重 (Pre-emphasis) 的目的和公式。

> **参考答案**：预加重的目的是提升高频分量，补偿声道对高频的衰减（声道相当于低通滤波器），使信号频谱更加平坦，有利于后续处理。公式为 $y[n] = x[n] - \alpha \cdot x[n-1]$，其中 $\alpha$ 通常取 0.95-0.97。

**Q9.** 比较 MFCC 和 Mel Spectrogram 的区别与各自适用场景。

> **参考答案**：MFCC 在 Mel Spectrogram 基础上多做了一步 DCT，将二维特征压缩为一维（通常13-39维），去除了各 Mel 通道间的相关性，适合传统模型如 GMM-HMM。Mel Spectrogram 保留了更多信息（二维时频表示），适合深度学习模型如 Whisper。现代语音模型通常使用 Mel Spectrogram。